# Pegasus CNN DailyMail

## Introduction
This notebook demonstrates **Pegasus CNN DailyMail** for text summarization. This pretrained checkpoint generates a domain-adapted abstractive summary.

### Objectives
- Load the shared article from `article.txt`.
- Generate a summary with this notebook's method.
- Evaluate the result against `reference_summary.txt` with shared ROUGE and BLEU metrics.

### Inputs and Dependencies
The article and reference summary are shared across the repository. Classical methods use the dependencies listed in `requirements.txt`; pretrained inference also downloads its model checkpoint.


In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("HF_HOME", str(ROOT / ".model_cache"))
os.environ.setdefault("NLTK_DATA", str(ROOT / ".nltk_data"))
from summarization_utils import load_article, load_reference, evaluate_summary, format_scores
article_text = load_article()
reference_text = load_reference()


**Pegasus-CNN/DailyMail** is a pre-trained language model introduced in 2019 that is designed for the task of summarizing news articles. It is a variant of the Pegasus model that is trained on the CNN/DailyMail dataset, which contains over 300,000 news articles and their corresponding summaries. Pegasus-CNN/DailyMail uses a transformer-based sequence-to-sequence model that is capable of generating high-quality summaries that capture the most important information from the original article. The model has achieved state-of-the-art performance on the CNN/DailyMail dataset and has also shown promising results on other summarization tasks. Like Pegasus-XSUM, Pegasus-CNN/DailyMail is available as part of the Hugging Face Transformers library.

In [2]:
try:
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    from summarization_utils import load_article, load_reference, evaluate_summary, format_scores

    article = load_article()
    reference_summary = load_reference()
    model_name = 'google/pegasus-cnn_dailymail'
    # The MultiNews checkpoint expects document separators; all other checkpoints receive the article as one document.
    model_input = article
    if model_name.endswith("multi_news"):
        model_input = " ||| ".join(part.strip() for part in article.split("\n\n") if part.strip())

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    max_input_tokens = max(1, min(1024, getattr(model.config, "max_position_embeddings", 1024) - 2))
    inputs = tokenizer(model_input, truncation=True, max_length=max_input_tokens, return_tensors="pt")
    with torch.inference_mode():
        generated = model.generate(**inputs, num_beams=4, max_length=120, min_length=20, no_repeat_ngram_size=3)
    summary = tokenizer.decode(generated[0], skip_special_tokens=True)
    print("Summary:", summary)
    print("Metrics:", format_scores(evaluate_summary(summary, reference_summary)))
except (OSError, RuntimeError, ImportError) as error:
    summary = None
    print(f"Pegasus inference unavailable in this environment: {type(error).__name__}: {error}")


C:\Users\tuhin\Downloads\Text-Summarization-Models-main\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


C:\Users\tuhin\Downloads\Text-Summarization-Models-main\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tuhin\Downloads\Text-Summarization-Models-main\.model_cache\hub\models--google--pegasus-cnn_dailymail. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Summary: In 2025, global temperatures remained exceptionally high .<n>NOAA’s National Centers for Environmental Information ranked the year as the third warmest in its records, which extend to 1850 .
Metrics: rouge1_f1=0.186, rouge2_f1=0.063, rougeL_f1=0.103, bleu4=0.033
